# PSL Live Translator: Project Overview

**Real-time Pakistan Sign Language (PSL) Urdu-alphabet recognizer**

Machine Learning (Lab) semester project, Sukkur IBA University
Authors: Aftab and Meraj

This notebook is the entry point of the submission. It explains what the project does, lists every notebook in the order it was written, shows the main result tables read directly from the `results/` folder, and says how to run the live app. It only **reads** saved files; it does not retrain anything or change any data.

## 1. What the project does

A webcam sees a hand. The app recognises which Urdu letter the hand is showing, adds it to a word on screen, and can read the word aloud.

```
Webcam frame -> MediaPipe HandLandmarker (21 points, 63 numbers)
             -> normalisation (wrist = origin, scaled by palm size)
             -> classical ML model -> letter + confidence
             -> hold the sign for 1 second -> letter is added to the word
```

- **36 static letters** (the hand holds still). Models compared: KNN, SVM, Random Forest, MLP. The chosen model uses 93 engineered landmark features + StandardScaler + SVC (C=100, gamma=0.01), trained with mirror augmentation.
- **4 dynamic letters** (Jeem, Aray, Alifmad, 2-Hay). These need movement, so the gesture is recorded for about 3 seconds and a Random Forest on 8 resampled time steps classifies it.
- **Tuey / Daal** look almost identical in landmarks. The app asks the user to pick one when the model cannot separate them (notebook 16 shows why).
- Only classical ML was used (no deep learning), as the course requires.

## 2. Dataset

The **UAlpha40** dataset (Pakistan Sign Language alphabet images and videos) is **not included** in this zip, as instructed. Download it from:

**https://data.mendeley.com/datasets/3pvnnckxyb/3**

License: **CC BY 4.0**. Credit: UAlpha40 dataset authors, Mendeley Data.

To re-run the notebooks that read raw images or videos, put the dataset under `data/raw/`. Everything else works from the saved files in `data/processed/`, `models/` and `results/`.

## 3. Notebook index

| # | Notebook | What it does | Needs the raw dataset? |
|---|---|---|---|
| 01 | `01_dataset_overview` | First look at the dataset: classes, image counts, signers | Yes |
| 02 | `02_extract_landmarks` | Runs MediaPipe on every image and saves 21 hand points per image to `landmarks.csv` | Yes |
| 03 | `03_eda` | Exploratory analysis of the landmark data | No |
| 04 | `04_split` | Signer-wise train/test split and leakage audit; writes `train.csv` and `test.csv` | No (but it rewrites the CSVs) |
| 05 | `05_models` | Compares KNN, SVM, Random Forest and MLP | No |
| 06 | `06_evaluation` | Evaluation of the chosen model | No |
| 07 | `07_error_analysis` | Which letters are confused with which | No |
| 08 | `08_test_predict` | Quick prediction check with the saved model | No |
| 09 | `09_left_hand` | Left-hand experiment (mirroring) | No |
| 10 | `10_features_angles` | Engineered features (angles and distances) | No |
| 11 | `11_clustering` | K-means clustering of letters | No |
| 12 | `12_feature_importance` | Which features matter most | No |
| 13 | `13_augmented_experiment` | Training with augmented images (model v4) | Yes (reads `aug_landmarks.csv`) |
| 14 | `14_summary_tables` | Summary tables of all model variants | No |
| 15 | `15_dynamic_letters` | Dynamic letters from videos, Random Forest on 8 time steps | Yes (reads `video_landmarks.csv`) |
| 16 | `16_tuey_daal_pair` | Why Tuey and Daal cannot be separated, and the group decision used in the app | No |

**Please do not re-run notebooks 01, 02, 04, 13 and 15.** They rewrite the landmark and split files or need the raw dataset and the larger augmented/video landmark files, which are not in this zip. Their saved outputs are kept inside each notebook. All other notebooks can be run top to bottom.

Model selection was done with signer-wise cross-validation on the training set. The test split was kept sealed and was not used to choose the model.

## 4. Check that the saved files are present

This cell only checks that the files the app and the notebooks need exist next to this notebook.

In [1]:
from pathlib import Path
import pandas as pd

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

needed = [
    "models/best_model_v3_eng.joblib",
    "models/dynamic_rf_T8.joblib",
    "models/hand_landmarker.task",
    "models/class_names.joblib",
    "data/processed/landmarks.csv",
    "data/processed/train.csv",
    "data/processed/test.csv",
    "app/live_app.py",
    "src/live.py",
    "app/assets/letters",
    "requirements.txt",
]
for rel in needed:
    print(("OK      " if (ROOT / rel).exists() else "MISSING ") + rel)

OK      models/best_model_v3_eng.joblib
OK      models/dynamic_rf_T8.joblib
OK      models/hand_landmarker.task
OK      models/class_names.joblib
OK      data/processed/landmarks.csv
OK      data/processed/train.csv
OK      data/processed/test.csv
OK      app/live_app.py
OK      src/live.py
OK      app/assets/letters
OK      requirements.txt


## 5. Main results

The tables below are read from the `results/` folder, so they always match the saved experiments.

In [2]:
def show(title, filename):
    path = ROOT / "results" / filename
    print(f"--- {title}  ({filename})")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("file not found")

show("Cross-validation of the four model families", "cv_results.csv")
show("Hyper-parameter tuning", "tuning_results.csv")
show("Core model on the sealed test split", "final_results.csv")

--- Cross-validation of the four model families  (cv_results.csv)


,Model,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Average,Std Dev
0,Baseline (majority class),0.0300,0.0300,0.0300,0.0323,0.0300,0.0305,0.0010
1,KNN,0.8060,0.8406,0.8522,0.8776,0.7483,0.8249,0.0500
2,SVM (RBF),0.8430,0.8799,0.8822,0.9261,0.8152,0.8693,0.0422
3,Random Forest,0.8707,0.8891,0.8938,0.9261,0.8453,0.8850,0.0299
4,MLP,0.9145,0.9169,0.9215,0.9376,0.8776,0.9136,0.0221


--- Hyper-parameter tuning  (tuning_results.csv)


,Model,Default CV Accuracy,Tuned CV Accuracy,Best Hyperparameters,Improvement
0,SVM (RBF),0.8693,0.9219,"{'model__C': 100, 'model__gamma': 0.01}",0.0526
1,MLP,0.9136,0.9201,"{'model__hidden_layer_sizes': (100, 50), 'mode...",0.0065
2,Random Forest,0.8850,0.8868,"{'model__max_depth': 20, 'model__n_estimators'...",0.0018
3,KNN,0.8249,0.8277,{'model__n_neighbors': 3},0.0028


--- Core model on the sealed test split  (final_results.csv)


,Model,Accuracy,Precision (macro),Recall (macro),F1 (macro)
0,SVM (RBF),0.9502,0.9524,0.9485,0.9482
1,MLP,0.9377,0.9396,0.9358,0.9361
2,Random Forest,0.9146,0.9165,0.9128,0.9129
3,KNN,0.8826,0.8859,0.8811,0.8810
4,Baseline,0.0285,0.0008,0.0278,0.0015


In [3]:
show("Model variants: signer-wise cross-validation", "summary_signer_cv_variants.csv")
show("Model variants: test split", "summary_test_variants.csv")

--- Model variants: signer-wise cross-validation  (summary_signer_cv_variants.csv)


,variant,right,left
0,"A raw63, no mirror (core style)",0.9252,0.7725
1,B raw63 + mirror (v2),0.9311,0.9311
2,C eng93 + mirror (v3),0.9499,0.9499
3,"D eng93, no mirror",0.9458,0.8871
4,F v4: eng93 + mirror + augmented,0.9509,0.9509


--- Model variants: test split  (summary_test_variants.csv)


,model,right,left
0,"core v1 (63, no mirror)",0.9502,0.7972
1,v2 (63 + mirror),0.9377,0.9377
2,"v3 (93 + mirror, g=0.01) CHOSEN",0.9324,0.9324
3,"v3b (93 + mirror, g=0.0068)",0.9342,0.9342
4,v4 (93 + mirror + augmented),0.9448,0.9448


In [4]:
show("Dynamic letters: cross-validation", "advanced_video_cv.csv")
show("Dynamic letters: held-out test videos", "advanced_video_test.csv")
show("Live webcam test of dynamic letters: confidence threshold", "live_dynamic_threshold.csv")

--- Dynamic letters: cross-validation  (advanced_video_cv.csv)


,T,model,n_features,acc,macro_f1
0,8,Baseline,520,0.271012,0.106604
1,8,KNN (k=3),520,0.966978,0.966609
2,8,SVM RBF,520,0.975982,0.975602
3,8,RF,520,0.990998,0.990597
4,8,MLP,520,0.985028,0.984553
5,16,Baseline,1040,0.271012,0.106604
6,16,KNN (k=3),1040,0.957977,0.957588
7,16,SVM RBF,1040,0.969875,0.969169
8,16,RF,1040,0.987921,0.987358
9,16,MLP,1040,0.979190,0.978731


--- Dynamic letters: held-out test videos  (advanced_video_test.csv)


,T,model,test_acc,test_macro_f1
0,8,Baseline,0.245902,0.098684
1,8,KNN (k=3),0.967213,0.967036
2,8,SVM RBF,0.983607,0.983165
3,8,RF,0.967213,0.966063
4,8,MLP,0.983607,0.983165
5,16,Baseline,0.245902,0.098684
6,16,KNN (k=3),0.950820,0.951304
7,16,SVM RBF,0.983607,0.983165
8,16,RF,0.967213,0.966063
9,16,MLP,0.983607,0.983165


--- Live webcam test of dynamic letters: confidence threshold  (live_dynamic_threshold.csv)


,threshold,asli_qubool_pct,kuch_nahi_jura_pct
0,0.3,100,100
1,0.4,98,80
2,0.5,86,50
3,0.6,70,40
4,0.7,59,10
5,0.8,34,10
6,0.9,11,10


### Headline numbers

- **Static letters (36 classes):** about **95.2%** signer-wise cross-validation accuracy.
- **Dynamic letters (studio videos):** **99.1%** cross-validation and **96.7%** on the held-out test videos.
- **Live dynamic test (own webcam):** 56 of 56 real gestures were recognised correctly. In a separate "no gesture" test, 4 of 10 attempts were wrongly accepted, so the app only answers when confidence is at least 0.60. Details are in `results/live_dynamic_log.csv` and `results/live_dynamic_threshold.csv`.
- **Tuey / Daal:** about 45% of all cross-validation errors; pair accuracy only about 62%. This is a limit of the landmark data, not of the model (notebook 16).

## 6. Run the live app

Use the same Python version the project was developed with (Python 3.12), in a fresh virtual environment:

```
python -m venv .venv
.venv\Scripts\activate
python -m pip install -r requirements.txt
python -m streamlit run app/live_app.py
```

Open the link Streamlit prints, press **START** and allow the camera. The first "Sunao" (speak) needs internet because the audio is made with gTTS.

How to use it:
- **Static letters:** hold the sign still for about 1 second; the letter is added to the word.
- **Dynamic letters:** press **Record**, wait for the 3-2-1 countdown, then perform the gesture. An optional **Auto mode** (Settings, off by default) tries to detect the gesture from hand motion; it is experimental.
- The sidebar **Harf guide** shows a photo of every letter.

## 7. Known limitations

- Tuey and Daal cannot be told apart from landmarks alone; the app asks the user to choose.
- About 5.8% of dataset hands are not detected by MediaPipe (for example closed fists).
- All training data was recorded in a studio, so other lighting, backgrounds and signers may lower accuracy.
- The test set is small (about +/-1.8 percentage points) and there is no external test set.
- There is no "no letter" class, so a non-sign hand shape can still be predicted as some letter. The confidence threshold and the one-second hold reduce this.
- Auto mode for dynamic letters is experimental and can trigger on small hand movements.
- Only 36 static and 4 dynamic letters are covered, not words or sentences.